# 01 Data Quality and Leakage Audit

Validate raw data health, schema integrity, fraud rarity, legacy-rule behavior, and leakage controls before modeling.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path(r"D:\Project\Data Science\financial_fraud_detection_&_transaction_intelligence")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
import json

import pandas as pd

from src.config import RAW_DATA_PATH, REPORT_DIR
from src.data import compute_quality_report

In [3]:
quality_path = REPORT_DIR / "data_quality_summary.json"
quality = (
    json.loads(quality_path.read_text(encoding="utf-8"))
    if quality_path.exists()
    else compute_quality_report(RAW_DATA_PATH)
)

pd.DataFrame(
    [
        {"Metric": "Rows", "Value": f"{quality['row_count']:,}"},
        {"Metric": "Columns", "Value": quality["column_count"]},
        {"Metric": "Missing values", "Value": quality["total_missing_values"]},
        {"Metric": "Duplicate rows", "Value": quality["duplicate_rows_hash_check"]},
        {"Metric": "Fraud rate", "Value": f"{quality['fraud_rate']:.4%}"},
        {"Metric": "Expected schema match", "Value": quality["expected_columns_match"]},
    ]
)

,Metric,Value
0,Rows,"6,362,620"
1,Columns,11
2,Missing values,0
3,Duplicate rows,0
4,Fraud rate,0.1291%
5,Expected schema match,True


In [4]:
type_summary = pd.DataFrame(quality["transaction_type_summary"])
type_summary.assign(fraud_rate_pct=(type_summary["fraud_rate"] * 100).round(4))

,type,transactions,fraud_transactions,fraud_rate,fraud_rate_pct
0,CASH_IN,1399284,0,0.000000,0.0000
1,CASH_OUT,2237500,4116,0.001840,0.1840
2,DEBIT,41432,0,0.000000,0.0000
3,PAYMENT,2151495,0,0.000000,0.0000
4,TRANSFER,532909,4097,0.007688,0.7688


In [5]:
pd.DataFrame(
    [
        {"Control": "Target excluded from features", "Status": quality["leakage_audit"]["target_column_excluded_from_features"]},
        {"Control": "Legacy flag benchmarked separately", "Status": quality["leakage_audit"]["isFlaggedFraud_treated_as_legacy_rule"]},
        {"Control": "Account IDs not category-encoded", "Status": quality["leakage_audit"]["identifier_columns_used_only_for_behavioral_flags"]},
        {"Control": "Validation strategy", "Status": quality["leakage_audit"]["validation_strategy"]},
    ]
)

,Control,Status
0,Target excluded from features,True
1,Legacy flag benchmarked separately,True
2,Account IDs not category-encoded,True
3,Validation strategy,step-aware split in the modeling pipeline


## Limitation Statement

This dataset is simulated. There is no real customer identity, calendar timestamp, investigation outcome, recovery amount, or production monitoring telemetry. Business impact should be framed as scenario analysis only.